# Feature engineering for the "Predicting Electric Vehicle Purchases" (Playground Series S6E9) dataset.

Findings from 01_EDA.ipynb that drove these features:
  - No missing values, no duplicates, no train/test distribution shift
    (KS tests all non-significant) -> we can engineer freely without
    worrying about leakage from imputation or shift correction.
  - Target is imbalanced: ~17.5% positive (Will_Buy_EV == 1).
  - Strongest single-feature signal (mutual information):
        Subsidy_Available            (cat, MI 0.162, Cramer's V 0.342)
        Environmental_Concern_Level  (num, MI 0.148, |Pearson| 0.46)
        Home_Charging_Possible       (cat, MI 0.100)
        Range_Anxiety_Level          (cat, MI 0.080, ordinal: Low>Medium>High
                                       purchase rate collapses from 18.9%->0.14%)
        Annual_Income_USD            (num, MI 0.054)
        City_Type                    (cat, MI 0.051)
  - Weak signal: Charging_Stations_Near_Home/Work, Daily_Commute_km, Age,
    Number_of_Cars_Owned individually, but they may still help in
    interactions/ratios and in tree-based models.

In [2]:
import os
import numpy as np
import pandas as pd

# Paths / constants

In [3]:
TRAIN_PATH = "./dataset/train.csv"
TEST_PATH = "./dataset/test.csv"
TRAIN_FE_PATH = "./dataset/train_fe.csv"
TEST_FE_PATH = "./dataset/test_fe.csv"

TARGET = "addicted_label"
ID_COL = "id"

NUM_COLS = [
    "age", "daily_screen_time_hours", "social_media_hours", "gaming_hours",
    "work_study_hours", "sleep_hours", "notifications_per_day",
    "app_opens_per_day", "weekend_screen_time",
]
CAT_COLS = ["gender", "stress_level", "academic_work_impact"]

STRESS_MAP = {"Low": 0, "Medium": 1, "High": 2}
YES_NO_MAP = {"Yes": 1, "No": 0}

# Loading

In [4]:
def load_raw_data(train_path=TRAIN_PATH, test_path=TEST_PATH):
    """Load raw train/test csv files and map the target to 0/1."""
    train = pd.read_csv(train_path)
    test = pd.read_csv(test_path)

    if train[TARGET].dtype == object:
        train[TARGET] = train[TARGET].map(YES_NO_MAP).astype(int)

    return train, test

# Feature engineering

In [5]:
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add engineered features on top of the raw columns. Safe to call on
    train and test separately - everything here is row-wise/deterministic,
    no target leakage. Missing raw values are preserved as NaN; imputation
    (where needed) happens per-model, not here, since CatBoost/XGBoost/
    LightGBM handle NaN natively while Logistic Regression does not.
    """
    df = df.copy()
 
    # missingness features (test whether "didn't answer" itself signals) 
    df["missing_count"] = df[NUM_COLS + CAT_COLS].isnull().sum(axis=1)
    df["has_any_missing"] = (df["missing_count"] > 0).astype(int)
 
    # ordinal / binary encodings (kept alongside originals for CatBoost/LGBM) 
    df["stress_ord"] = df["stress_level"].map(STRESS_MAP)
    df["academic_impact_bin"] = df["academic_work_impact"].map(YES_NO_MAP)
 
    # ratios / interactions on the behavioral numeric columns 
    df["screen_to_sleep_ratio"] = df["daily_screen_time_hours"] / (df["sleep_hours"] + 1)
    df["social_to_gaming_ratio"] = df["social_media_hours"] / (df["gaming_hours"] + 1)
    df["notifications_per_app_open"] = df["notifications_per_day"] / (df["app_opens_per_day"] + 1)
    df["weekend_vs_weekday_diff"] = df["weekend_screen_time"] - df["daily_screen_time_hours"]
    df["leisure_screen_hours"] = df["daily_screen_time_hours"] - df["work_study_hours"]
    df["sleep_deficit"] = (8 - df["sleep_hours"]).clip(lower=0)
    df["screen_per_notification"] = df["daily_screen_time_hours"] / (df["notifications_per_day"] + 1)
 
    # simple flags 
    df["high_stress_flag"] = (df["stress_level"] == "High").astype(float)
    df["low_sleep_flag"] = (df["sleep_hours"] < 6).astype(float)
    df["heavy_screen_flag"] = (df["daily_screen_time_hours"] > df["daily_screen_time_hours"].median()).astype(float)
 
    return df

def get_feature_lists(df: pd.DataFrame):
    """
    Return (num_cols, cat_cols) after engineering. Raw categorical columns
    stay categorical (for CatBoost/LightGBM native handling); engineered
    binary/ordinal/ratio features are numeric.
    """
    engineered_num = [
        "missing_count", "has_any_missing", "stress_ord", "academic_impact_bin",
        "screen_to_sleep_ratio", "social_to_gaming_ratio", "notifications_per_app_open",
        "weekend_vs_weekday_diff", "leisure_screen_hours", "sleep_deficit",
        "screen_per_notification", "high_stress_flag", "low_sleep_flag", "heavy_screen_flag",
    ]
    num_cols = NUM_COLS + [c for c in engineered_num if c in df.columns]
    cat_cols = [c for c in CAT_COLS if c in df.columns]
    return num_cols, cat_cols

# Entry point

In [6]:
def main():
    os.makedirs("./dataset", exist_ok=True)
 
    train, test = load_raw_data()
    train_fe = engineer_features(train)
    test_fe = engineer_features(test)
 
    num_cols, cat_cols = get_feature_lists(train_fe)
 
    print("=" * 70)
    print("FEATURE ENGINEERING SUMMARY")
    print("=" * 70)
    print(f"Original columns : {train.shape[1]}")
    print(f"Engineered columns: {train_fe.shape[1]}")
    print(f"Numeric features ({len(num_cols)}): {num_cols}")
    print(f"Categorical features ({len(cat_cols)}): {cat_cols}")
 
    train_fe.to_csv(TRAIN_FE_PATH, index=False)
    test_fe.to_csv(TEST_FE_PATH, index=False)
    print(f"\nSaved: {TRAIN_FE_PATH}")
    print(f"Saved: {TEST_FE_PATH}")

In [7]:
if __name__ == "__main__":
    main()

FEATURE ENGINEERING SUMMARY
Original columns : 14
Engineered columns: 28
Numeric features (23): ['age', 'daily_screen_time_hours', 'social_media_hours', 'gaming_hours', 'work_study_hours', 'sleep_hours', 'notifications_per_day', 'app_opens_per_day', 'weekend_screen_time', 'missing_count', 'has_any_missing', 'stress_ord', 'academic_impact_bin', 'screen_to_sleep_ratio', 'social_to_gaming_ratio', 'notifications_per_app_open', 'weekend_vs_weekday_diff', 'leisure_screen_hours', 'sleep_deficit', 'screen_per_notification', 'high_stress_flag', 'low_sleep_flag', 'heavy_screen_flag']
Categorical features (3): ['gender', 'stress_level', 'academic_work_impact']

Saved: ./dataset/train_fe.csv
Saved: ./dataset/test_fe.csv
